# 04 — Análisis semántico con BETO
**Proyecto: reseñas de cine y series** · Corpus consolidado exportado de MongoDB

Este notebook realiza polisemia, Masked Language Modeling, embeddings BETO `[CLS]` y búsqueda semántica. **No requiere conexión a MongoDB**: utiliza `resenas_mongodb.csv`. Los resultados no se han ejecutado aquí; se generan al correr las celdas en Colab.

> En Colab, elegir **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** (si está disponible).

## 1. Dependencias y archivos

In [1]:
!pip -q install "transformers>=4.40,<5" torch pandas numpy scikit-learn tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import transformers
import huggingface_hub
import torch

print("Transformers:", transformers.__version__)
print("Hugging Face:", huggingface_hub.__version__)
print("GPU disponible:", torch.cuda.is_available())

Transformers: 4.57.6
Hugging Face: 0.36.2
GPU disponible: True


In [6]:
import os, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display

#   embeddings_beto.py y resenas_mongodb.csv
if not (Path('embeddings_beto.py').exists() and Path('resenas_mongodb.csv').exists()):
    from google.colab import files
    files.upload()

assert Path('embeddings_beto.py').exists(), 'Falta embeddings_beto.py'
assert Path('resenas_mongodb.csv').exists(), 'Falta resenas_mongodb.csv'
print('Archivos disponibles.')

Saving resenas_mongodb.csv to resenas_mongodb (1).csv
Saving embeddings_beto.py to embeddings_beto.py
Archivos disponibles.


## 2. Modelo BETO

In [7]:
import embeddings_beto as eb

tokenizer, modelo, dispositivo = eb.cargar_beto()
print('Modelo:', eb.MODELO)
print('Dispositivo:', dispositivo)
print('Dimensión:', modelo.config.hidden_size)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modelo: dccuchile/bert-base-spanish-wwm-cased
Dispositivo: cuda
Dimensión: 768


## 3. Corpus consolidado: datos de Proyecto 1 + scraping

In [8]:
df = eb.cargar_resenas('resenas_mongodb.csv')
requeridas = ['id_resena','texto_es','genero_principal','polaridad']
assert all(c in df.columns for c in requeridas), 'Faltan columnas necesarias'
assert df.id_resena.is_unique, 'IDs duplicados'
print('Reseñas listas:', len(df))
if 'origen_texto' in df:
    print('Origen de los textos:')
    print(df.origen_texto.value_counts(dropna=False))
display(df[requeridas + (['origen_texto'] if 'origen_texto' in df else [])].head(3))

Reseñas listas: 12368
Origen de los textos:
origen_texto
traducido    6283
nativo       6085
Name: count, dtype: int64


,id_resena,texto_es,genero_principal,polaridad,origen_texto
0,p1_1,Vi esta película por primera vez cuando tenía ...,Animación,positiva,traducido
1,p1_2,Kingpin es una película divertida con algunas ...,Comedia,intermedia,traducido
2,p1_3,En la necesidad de un poco de espíritu navideñ...,Drama,positiva,traducido


## 4. Polisemia: misma palabra, distintos contextos
Comparamos la representación contextual de cada palabra mediante similitud coseno. Un valor bajo indica representaciones más distintas, **no prueba por sí solo comprensión semántica**.

In [9]:
ejemplos = {
    'papel': ['La actriz hizo un papel extraordinario en la película.', 'El protagonista encontró un papel con una carta misteriosa.'],
    'plano': ['El director utilizó un plano muy cerrado del rostro.', 'El personaje vivía en un terreno plano y desértico.'],
    'estrella': ['La estrella de cine recibió muchos premios.', 'Una estrella brillante apareció en el cielo nocturno.'],
    'corto': ['El cortometraje fue muy corto pero emocionante.', 'El protagonista llevaba el cabello corto.'],
    'crítica': ['La crítica de cine elogió la actuación principal.', 'La situación de la protagonista era crítica.']
}
resultado_polisemia = eb.analizar_polisemia(ejemplos, tokenizer, modelo, dispositivo)
display(resultado_polisemia)
resultado_polisemia.to_csv('polisemia_beto.csv', index=False, encoding='utf-8-sig')

,palabra,contexto_1,contexto_2,similitud_coseno
0,papel,La actriz hizo un papel extraordinario en la p...,El protagonista encontró un papel con una cart...,0.6974
1,plano,El director utilizó un plano muy cerrado del r...,El personaje vivía en un terreno plano y desér...,0.7752
2,estrella,La estrella de cine recibió muchos premios.,Una estrella brillante apareció en el cielo no...,0.7669
3,corto,El cortometraje fue muy corto pero emocionante.,El protagonista llevaba el cabello corto.,0.7812
4,crítica,La crítica de cine elogió la actuación principal.,La situación de la protagonista era crítica.,0.6310


## 5. Masked Language Modeling
Las probabilidades corresponden a predicciones de tokens, no a exactitud global del modelo.

In [10]:
frases_mask = [
    'La película fue [MASK].',
    'La actuación del protagonista fue [MASK].',
    'Los efectos especiales son [MASK].',
    'Esta película de terror fue [MASK].',
    'Esta película de comedia fue [MASK].',
    'El final de la historia fue [MASK].'
]
resultado_mask = eb.completar_mascaras(frases_mask, top_k=5, dispositivo=dispositivo)
display(resultado_mask)
resultado_mask.to_csv('masked_lm_beto.csv', index=False, encoding='utf-8-sig')

Device set to use cuda:0


Masked LM:   0%|          | 0/6 [00:00<?, ?it/s]

,frase,prediccion,probabilidad
0,La película fue [MASK].,estrenada,0.1110
1,La película fue [MASK].,titulada,0.0859
2,La película fue [MASK].,prohibida,0.0766
3,La película fue [MASK].,de,0.0390
4,La película fue [MASK].,distribuida,0.0389
5,La actuación del protagonista fue [MASK].,excelente,0.0897
6,La actuación del protagonista fue [MASK].,buena,0.0534
7,La actuación del protagonista fue [MASK].,escasa,0.0415
8,La actuación del protagonista fue [MASK].,fantástica,0.0371
9,La actuación del protagonista fue [MASK].,notable,0.0339


In [11]:
frases_contexto = [
    'La película me hizo llorar de emoción. Fue una experiencia [MASK].',
    'La película fue tan aburrida que casi me duermo. Fue [MASK].',
    'El público no dejó de reír durante la película. La comedia fue [MASK].',
    'Las escenas oscuras y los sonidos aterradores provocaron miedo. La película fue [MASK].',
    'Los actores interpretaron sus personajes de manera convincente. La actuación fue [MASK].',
    'La historia tenía muchos errores y un final sin sentido. El guion fue [MASK].'
]
resultado_contexto = eb.completar_mascaras(frases_contexto, top_k=5, dispositivo=dispositivo)
display(resultado_contexto)
resultado_contexto.to_csv('masked_lm_contexto_beto.csv', index=False, encoding='utf-8-sig')

Device set to use cuda:0


Masked LM:   0%|          | 0/6 [00:00<?, ?it/s]

,frase,prediccion,probabilidad
0,La película me hizo llorar de emoción. Fue una...,increíble,0.3514
1,La película me hizo llorar de emoción. Fue una...,maravillosa,0.1649
2,La película me hizo llorar de emoción. Fue una...,única,0.0547
3,La película me hizo llorar de emoción. Fue una...,extraordinaria,0.0370
4,La película me hizo llorar de emoción. Fue una...,terrible,0.0352
5,La película fue tan aburrida que casi me duerm...,divertido,0.1561
6,La película fue tan aburrida que casi me duerm...,increíble,0.1526
7,La película fue tan aburrida que casi me duerm...,horrible,0.1012
8,La película fue tan aburrida que casi me duerm...,genial,0.0766
9,La película fue tan aburrida que casi me duerm...,terrible,0.0492


## 6. Embeddings del corpus completo
BETO produce vectores de 768 dimensiones. Se usan lotes lógicos de 100 y sublotes de 10 para controlar la memoria GPU. Los textos se truncan a 256 tokens. **La matriz y los metadatos conservan exactamente el mismo orden de filas.**

Si ya existe `vectores_beto_final.npy` de una ejecución completa, se reutiliza solo cuando coincide su cantidad de filas con los metadatos guardados y los IDs actuales.

In [12]:
ruta_vec = Path('vectores_beto_final.npy')
ruta_meta = Path('metadata_beto_final.csv')
reutilizar = False
if ruta_vec.exists() and ruta_meta.exists():
    ids_previos = pd.read_csv(ruta_meta, usecols=['id_resena'], dtype={'id_resena':str})['id_resena']
    forma = np.load(ruta_vec, mmap_mode='r').shape
    reutilizar = (forma == (len(df), modelo.config.hidden_size)
                  and ids_previos.tolist() == df['id_resena'].astype(str).tolist())
if reutilizar:
    matriz_beto_final = np.load(ruta_vec)
    print('Embeddings existentes reutilizados.')
else:
    matriz_beto_final = eb.vectores_beto(
        textos=df['texto_es'].tolist(), tokenizer=tokenizer, modelo=modelo,
        dispositivo=dispositivo, lote=100, sublote=10
    )
    np.save(ruta_vec, matriz_beto_final)
    df.to_csv(ruta_meta, index=False, encoding='utf-8-sig')
print('Matriz:', matriz_beto_final.shape)
assert matriz_beto_final.shape == (len(df), 768)

Lotes de 100:   0%|          | 0/124 [00:00<?, ?it/s]

Matriz: (12368, 768)


## 7. Búsqueda semántica sobre todas las reseñas
Evaluar manualmente la relevancia: una similitud coseno alta no garantiza coincidencia de sentimiento.

In [13]:
consultas = [
    'Una película emocionante con una historia triste',
    'Una película de terror con mucho suspenso y escenas aterradoras',
    'Una comedia muy divertida que me hizo reír',
    'Una película aburrida con una historia lenta y malas actuaciones'
]
resultados = []
for consulta in consultas:
    encontrados = eb.buscar_resenas(
        consulta, df, matriz_beto_final, tokenizer, modelo, dispositivo, top=5
    ).copy()
    encontrados.insert(0, 'consulta', consulta)
    resultados.append(encontrados)
    print('CONSULTA:', consulta)
    display(encontrados)
resultados_finales = pd.concat(resultados, ignore_index=True)
resultados_finales.to_csv('busquedas_semanticas_beto_final.csv', index=False, encoding='utf-8-sig')

Lotes de 100:   0%|          | 0/1 [00:00<?, ?it/s]

CONSULTA: Una película emocionante con una historia triste


,consulta,id_resena,similitud,genero_principal,polaridad,texto_es
11136,Una película emocionante con una historia triste,sensacine_150411,0.9430,Terror,positiva,Una película muy miedosa
9116,Una película emocionante con una historia triste,sensacine_1005277445,0.9289,Comedia,positiva,"Una película romántica e interesante, llena de..."
10204,Una película emocionante con una historia triste,sensacine_424916,0.9255,Acción,negativa,una pelicula inpredecible basada en un libro c...
8886,Una película emocionante con una historia triste,sensacine_1026586723,0.9214,Comedia,negativa,Aburrida sin sentido ridícula Es una película ...
12334,Una película emocionante con una historia triste,sensacine_1038007537,0.9196,Animación,positiva,"Una película adelantada a su época, simplement..."


Lotes de 100:   0%|          | 0/1 [00:00<?, ?it/s]

CONSULTA: Una película de terror con mucho suspenso y escenas aterradoras


,consulta,id_resena,similitud,genero_principal,polaridad,texto_es
9116,Una película de terror con mucho suspenso y es...,sensacine_1005277445,0.9434,Comedia,positiva,"Una película romántica e interesante, llena de..."
10360,Una película de terror con mucho suspenso y es...,sensacine_1001156827,0.9225,Terror,positiva,"Una película de terror muy bien dirigida, con ..."
10204,Una película de terror con mucho suspenso y es...,sensacine_424916,0.9103,Acción,negativa,una pelicula inpredecible basada en un libro c...
11537,Una película de terror con mucho suspenso y es...,sensacine_1021451197,0.9103,Animación,positiva,Salvo un par de bromas descontextualizadas y u...
11101,Una película de terror con mucho suspenso y es...,sensacine_147878,0.9095,Terror,positiva,"Interesante pelicula,diferente e inquietante a..."


... (2 outputs left out)

,consulta,id_resena,similitud,genero_principal,polaridad,texto_es
11699,Una comedia muy divertida que me hizo reír,sensacine_130599,0.9050,Animación,positiva,Me encantó y me reí un monntón
12132,Una comedia muy divertida que me hizo reír,sensacine_1040372073,0.8947,Animación,intermedia,"Su quería verla, me ha gustado"
8590,Una comedia muy divertida que me hizo reír,sensacine_1416012,0.8893,Comedia,positiva,a mí me pareció una película muy entrañable y ...
7264,Una comedia muy divertida que me hizo reír,sensacine_1007011082,0.8852,Drama,positiva,Me encanta esta película y quiero volver a ver...
11142,Una comedia muy divertida que me hizo reír,sensacine_150045,0.8835,Terror,negativa,Una pelicula penosa


Lotes de 100:   0%|          | 0/1 [00:00<?, ?it/s]

CONSULTA: Una película aburrida con una historia lenta y malas actuaciones


,consulta,id_resena,similitud,genero_principal,polaridad,texto_es
9116,Una película aburrida con una historia lenta y...,sensacine_1005277445,0.9412,Comedia,positiva,"Una película romántica e interesante, llena de..."
11136,Una película aburrida con una historia lenta y...,sensacine_150411,0.9195,Terror,positiva,Una película muy miedosa
7272,Una película aburrida con una historia lenta y...,sensacine_1004455777,0.9161,Drama,negativa,"Un inicio de película asqueroso, muy mala, el ..."
10204,Una película aburrida con una historia lenta y...,sensacine_424916,0.9137,Acción,negativa,una pelicula inpredecible basada en un libro c...
12334,Una película aburrida con una historia lenta y...,sensacine_1038007537,0.9054,Animación,positiva,"Una película adelantada a su época, simplement..."


## 8. Exportar entregables
El ZIP incluye embeddings, metadatos, polisemia, Masked LM y búsquedas. Descargalo antes de cerrar Colab.

In [14]:
archivos = [
    'vectores_beto_final.npy', 'metadata_beto_final.csv',
    'polisemia_beto.csv', 'masked_lm_beto.csv',
    'masked_lm_contexto_beto.csv', 'busquedas_semanticas_beto_final.csv'
]
for archivo in archivos:
    assert Path(archivo).exists(), f'Falta {archivo}; ejecutá su celda antes de exportar.'
with zipfile.ZipFile('BETO_RESULTADOS_FINALES.zip', 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for archivo in archivos:
        z.write(archivo)
print('ZIP listo:', Path('BETO_RESULTADOS_FINALES.zip').stat().st_size, 'bytes')
from google.colab import files
files.download('BETO_RESULTADOS_FINALES.zip')

ZIP listo: 43059635 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>